<a href="https://colab.research.google.com/github/sameerahmed-cloud/text2sql-small-llm/blob/main/02_training_data.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Cell 1: get the latest code (same as your other notebooks)
import os
REPO_DIR = "/content/text2sql-small-llm"
if not os.path.exists(REPO_DIR):
    !git clone https://github.com/sameerahmed-cloud/text2sql-small-llm.git {REPO_DIR}
else:
    !git -C {REPO_DIR} pull
%cd {REPO_DIR}

Already up to date.
/content/text2sql-small-llm


In [2]:
# Cell 2: Drive + all datasets, including BIRD train
# HF_TOKEN must be in Colab Secrets with "Notebook access" on.
# BIRD train (8.9 GB) comes from your HF repo, so the first unpack takes a while.
from google.colab import drive
drive.mount("/content/drive")

from src.get_data import unpack_all
from src.data_prep import make_chat_files, check_chat_file, DRIVE_PROCESSED

paths = unpack_all(include_bird_train=True)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✓ Paths saved to /content/data/raw/paths.json


In [3]:
# Cell 3: filter + tidy + format (resumable: just re-run after a disconnect)
summary = make_chat_files(paths)
{k: {kk: vv for kk, vv in v.items() if kk != "dropped"} for k, v in summary.items()}

Gold check: 14127 already in gold_check_train.jsonl, 0 to run
Tidy check: 4978 of 13718 queries change; 200/200 sampled give identical results
  train: formatted 2000/13718
  train: formatted 4000/13718
  train: formatted 6000/13718
  train: formatted 8000/13718
  train: formatted 10000/13718
  train: formatted 12000/13718
✓ train: 13718 examples | excluded 847 (['retail_world', 'works_cycles']) | gold failed 36 {'no such column/table': 11, 'other error': 2, 'timeout': 23} | empty result (kept) 448 | tidied 4978 -> /content/drive/MyDrive/text2sql/data/processed/sft_train.jsonl (73 MB)
Gold check: 1634 already in gold_check_val.jsonl, 0 to run
Tidy check: 479 of 1628 queries change; 200/200 sampled give identical results
✓ val: 1628 examples | excluded 0 (['retail_world', 'works_cycles']) | gold failed 6 {'timeout': 6} | empty result (kept) 60 | tidied 479 -> /content/drive/MyDrive/text2sql/data/processed/sft_val.jsonl (10 MB)
Saved /content/drive/MyDrive/text2sql/data/processed/sft_sum

{'train': {'questions_in': 14601,
  'excluded_db': 847,
  'gold_failed': 36,
  'gold_failed_by_outcome': {'no such column/table': 11,
   'other error': 2,
   'timeout': 23},
  'gold_empty_result_kept': 448,
  'gold_tidied': 4978,
  'examples_out': 13718,
  'by_source': {'spider': 6202, 'bird_train': 7516},
  'file_mb': 73.1},
 'val': {'questions_in': 1634,
  'excluded_db': 0,
  'gold_failed': 6,
  'gold_failed_by_outcome': {'timeout': 6},
  'gold_empty_result_kept': 60,
  'gold_tidied': 479,
  'examples_out': 1628,
  'by_source': {'spider': 701, 'bird_train': 927},
  'file_mb': 10.4}}

In [4]:
# Cell 4: check them with the real Qwen chat template (tokenizer only, no model)
from transformers import AutoTokenizer
tok = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-Coder-1.5B-Instruct")
for name in ["train", "val"]:
    check_chat_file(DRIVE_PROCESSED / f"sft_{name}.jsonl", tok)

✓ /content/drive/MyDrive/text2sql/data/processed/sft_train.jsonl: 13761 rows; 5 spot-checked, training text = test prompt + answer

End of one example (what the model learns to produce comes after 'assistant'):
...d)
)

Question: Find all the songs performed by artist with last name "Heilo"

Write one SQLite query that answers the question. Return only the SQL, inside a ```sql code block, with no explanation.<|im_end|>
<|im_start|>assistant
```sql
SELECT T3.Title FROM Performance AS T1 JOIN Band AS T2 ON T1.bandmate = T2.id JOIN Songs AS T3 ON T3.SongId = T1.SongId WHERE T2.Lastname = "Heilo"
```<|im_end|>

✓ /content/drive/MyDrive/text2sql/data/processed/sft_val.jsonl: 1628 rows; 5 spot-checked, training text = test prompt + answer

End of one example (what the model learns to produce comes after 'assistant'):
...") REFERENCES `aircraft`(`Aircraft_ID`)
)
/* Sample values:
  ID: 1, 2, 3
  Airport_ID: 1, 2, 6
  Aircraft_ID: 2, 1, 5
*/

Question: List names of all pilot in descending ord

In [5]:
# Cell 5: why were examples dropped?
import pandas as pd
dropped = pd.DataFrame(summary["train"]["dropped"] + summary["val"]["dropped"])
print(dropped["outcome"].value_counts())
dropped[["id", "db_id", "outcome", "error", "seconds"]].head(20)

outcome
no such column/table    341
timeout                  29
other error               2
Name: count, dtype: int64


,id,db_id,outcome,error,seconds
0,spider-train-3153,assets_maintenance,no such column/table,no such table: Ref_Company_Types,0.00
1,spider-train-4513,document_management,other error,ORDER BY clause should come after INTERSECT no...,0.00
2,spider-train-4514,document_management,other error,ORDER BY clause should come after INTERSECT no...,0.00
3,bird_train-train-5,movie_platform,timeout,interrupted,30.03
4,bird_train-train-6,movie_platform,timeout,interrupted,30.08
5,bird_train-train-7,movie_platform,timeout,interrupted,30.00
6,bird_train-train-115,movie_platform,timeout,interrupted,30.02
7,bird_train-train-1048,talkingdata,timeout,interrupted,30.00
8,bird_train-train-1050,talkingdata,timeout,interrupted,30.01
9,bird_train-train-1052,talkingdata,timeout,interrupted,30.01


In [6]:
# Cell 6: which databases do the dropped questions come from?
dropped.groupby(["db_id", "outcome"]).size().sort_values(ascending=False).head(15)

,,0
db_id,outcome,
retail_world,no such column/table,330
retails,timeout,6
talkingdata,timeout,6
app_store,no such column/table,6
donor,timeout,5
shooting,no such column/table,4
movie_platform,timeout,4
bike_share_1,timeout,3
world_development_indicators,timeout,3


In [7]:
# Cell 7: how much of retail_world survived, and what do the errors say?
import pandas as pd
from src.data_prep import read_jsonl
gc = pd.DataFrame(read_jsonl(DRIVE_PROCESSED / "gold_check_train.jsonl"))
gc["db_id"] = gc["id"].map({c["id"]: c["db_id"] for c in read_jsonl(DRIVE_PROCESSED / "train.jsonl")})
rw = gc[gc["db_id"] == "retail_world"]
print(len(rw), "questions;", rw["ok"].sum(), "ran fine")
print(rw.loc[~rw["ok"], "error"].value_counts().head(8))

373 questions; 43 ran fine
error
no such table: Order Details          72
no such column: T2.CompanyName        49
no such table: EmployeeTerritories    35
no such table: Territories            20
no such column: CompanyName           15
no such column: T1.CompanyName        12
no such column: Salary                11
no such column: Title                  9
Name: count, dtype: int64
